# Syncing Apache Ranger Policies from Hive to Trino Service

This script demonstrates how to convert Hive Ranger policies to Trino. Please note the following constraints:

- Only policies with `database`, `table`, and `column` resources are processed.
- Only `select`, `update`, `create`, `drop`, and `alter` permissions are migrated.
- Masking and row-filter policies are ignored (only access policies are synced).
- Tested on Apache Ranger 2.8.0 (Docker).

In [1]:

from apache_ranger.client.ranger_client          import *
from apache_ranger.model.ranger_policy           import *
from apache_ranger.model.ranger_service          import *
from apache_ranger.model.ranger_service_resource import *
from apache_ranger.model.ranger_service_tags     import *
from apache_ranger.model.ranger_tagdef           import *
from apache_ranger.model.ranger_tag              import *
from datetime                                    import datetime


In [ ]:
ranger_url  = 'http://localhost:6080'
ranger_auth = ('admin', 'rangerR0cks!')
ranger = RangerClient(ranger_url, ranger_auth)

## Functions

In [ ]:
# Custom Errors
class NoPolicyItemsError(Exception):
  pass

class UnsupportedPolicyError(Exception):
  pass

def migrate_accesses_to_trino(accesses):
  accesses_trino=[]
  trino_policies = ['select', 'update', 'create', 'drop', 'alter']

  for ac in accesses:
    if ac['type'] in trino_policies:
      if ac['type'] == 'update':
        insert_ac = ac.copy()
        insert_ac['type'] = 'insert'
        delete_ac = ac.copy()
        delete_ac['type'] = 'delete'
        accesses_trino.append(insert_ac)
        accesses_trino.append(delete_ac)
        continue
      accesses_trino.append(ac)
  return accesses_trino

def convert_to_trino_resources(resources, catalog_name=""):
  trino_resources = dict()
  trino_resources['catalog'] = RangerPolicyResource({ 'values': [catalog_name] })
  if 'database' in resources.keys():
    trino_resources['schema'] =  RangerPolicyResource(resources['database'])
  if 'table' in resources.keys():
    trino_resources['table'] =  RangerPolicyResource(resources['table'])
  if 'column' in resources.keys():
    trino_resources['column'] =  RangerPolicyResource(resources['column'])
  return trino_resources

def create_trino_policy_from_hive(hive_policy, trino_service_name, catalog_name='hive', policy_prefix='hivesync_'): 

  retrieved_policy = hive_policy #54
  available_resources = ['table','database','column']
  # Check resource supports
  if not all([x in available_resources for x in retrieved_policy.resources.keys()]):
    raise UnsupportedPolicyError('Unsupported policy')

  policy             = RangerPolicy()
  policy.service     = trino_service_name
  policy.name        = f"{policy_prefix}{retrieved_policy['id']}_{retrieved_policy.version}"
  policy.description = retrieved_policy.description
  policy.resources   = convert_to_trino_resources(retrieved_policy.resources, catalog_name)
  policy.policyPriority = retrieved_policy.policyPriority
  policy.isAuditEnabled = retrieved_policy.isAuditEnabled
  policy.isDenyAllElse = retrieved_policy.isDenyAllElse

  policyItems = []
  denyPolicyItems = []
  allowExceptions = []
  denyExceptions = []

  if 'policyItems' in retrieved_policy.keys():
    for item in retrieved_policy.policyItems:
      policy_item = RangerPolicyItem()
      trino_accesses = migrate_accesses_to_trino(item['accesses'])
      if len(trino_accesses)==0:
        # No Trino policies remain after the access migration
        continue
      policy_item.accesses = trino_accesses
      policy_item.delegateAdmin = item['delegateAdmin']
      if 'users' in item.keys():
        policy_item.users = item['users']
      if 'roles' in item.keys():
        policy_item.roles = item['roles']
      if 'groups' in item.keys():
        policy_item.groups = item['groups']
      
      policyItems.append(policy_item)

    policy.policyItems = policyItems

  if 'denyPolicyItems' in retrieved_policy.keys():
    for item in retrieved_policy.denyPolicyItems:
      deny_policy_item = RangerPolicyItem()
      trino_accesses = migrate_accesses_to_trino(item['accesses'])
      if len(trino_accesses)==0:
        # No Trino policies remain after the access migration
        continue
      deny_policy_item.accesses = trino_accesses
      deny_policy_item.delegateAdmin = item['delegateAdmin']
      if 'users' in item.keys():
        deny_policy_item.users = item['users']
      if 'roles' in item.keys():
        deny_policy_item.roles = item['roles']
      if 'groups' in item.keys():
        deny_policy_item.groups = item['groups']
      denyPolicyItems.append(deny_policy_item)
    
    policy.denyPolicyItems = denyPolicyItems
    
  if 'allowExceptions' in retrieved_policy.keys():
    for item in retrieved_policy.allowExceptions:
      allow_exception_item = RangerPolicyItem()
      trino_accesses = migrate_accesses_to_trino(item['accesses'])
      if len(trino_accesses)==0:
        # No Trino policies remain after the access migration
        continue
      allow_exception_item.accesses = trino_accesses
      allow_exception_item.delegateAdmin = item['delegateAdmin']
      if 'users' in item.keys():
        allow_exception_item.users = item['users']
      if 'roles' in item.keys():
        allow_exception_item.roles = item['roles']
      if 'groups' in item.keys():
        allow_exception_item.groups = item['groups']
      allowExceptions.append(allow_exception_item)
    
    policy.allowExceptions = allowExceptions
  
  if 'denyExceptions' in retrieved_policy.keys():
    for item in retrieved_policy.denyExceptions:
      deny_exception_item = RangerPolicyItem()
      trino_accesses = migrate_accesses_to_trino(item['accesses'])
      if len(trino_accesses)==0:
        # No Trino policies remain after the access migration
        continue
      deny_exception_item.accesses = trino_accesses
      deny_exception_item.delegateAdmin = item['delegateAdmin']
      if 'users' in item.keys():
        deny_exception_item.users = item['users']
      if 'roles' in item.keys():
        deny_exception_item.roles = item['roles']
      if 'groups' in item.keys():
        deny_exception_item.groups = item['groups']
      denyExceptions.append(deny_exception_item)
    
    policy.denyExceptions = denyExceptions
  
  if 'validitySchedules' in retrieved_policy.keys():
    policy.validitySchedules = retrieved_policy.validitySchedules
  
  if len(policyItems)==0 and len(denyPolicyItems)==0 and len(allowExceptions)==0 and len(denyExceptions)==0:
    # This policy is invalid.
    raise NoPolicyItemsError("No Policy Items")

  return policy

In [ ]:
# hive_policy = ranger.get_policy_by_id(68)
# a  = create_trino_policy_from_hive(hive_policy, 'dev_trino')
# ranger.create_policy(a)

## Create Test Data

In [ ]:
target_service = 'dev_hive'

for i in range(2000):
  policy_name = f'test policy_{i}'

  policy             = RangerPolicy()
  policy.service     = target_service
  policy.name        = policy_name
  policy.description = 'test description'
  policy.resources   = { 'database': RangerPolicyResource({ 'values': [f'test_db_{i}'] }),
                        'table':    RangerPolicyResource({ 'values': ['test_tbl'] }),
                        'column':   RangerPolicyResource({ 'values': ['*'] }) }

  allowItem1          = RangerPolicyItem()
  allowItem1.users    = [ f'admin' ]
  allowItem1.accesses = [ RangerPolicyItemAccess({ 'type': 'create' }),
                          RangerPolicyItemAccess({ 'type': 'alter' }),
                          RangerPolicyItemAccess({ 'type': 'select' }) ]

  denyItem1          = RangerPolicyItem()
  denyItem1.users    = [ f'admin' ]
  denyItem1.accesses = [ RangerPolicyItemAccess({ 'type': 'drop' }) ]

  policy.policyItems     = [ allowItem1 ]
  policy.denyPolicyItems = [ denyItem1 ]

  created_policy = ranger.create_policy(policy)


## Sync Policies

In [ ]:
import json
import requests

def get_policies_from_ranger(url, auth, service_name):
  s = requests.Session()
  s.auth = auth
  s.headers = { 'Accept': 'application/json', 'Content-type': 'text/json' }
  a = s.get(f'{url}/service/plugins/policies/download/{service_name}').content
  return json.loads(a)

In [ ]:
# limits 200 policies
# hive_policies = ranger.get_policies_in_service('dev_hive') 

# OPTION-1 JSON
# with open('Hive_Ranger_Policies_20260331_185657.json', 'rb') as file:
#   hive_policies=json.load(file)


# with open('Trino_Ranger_Policies_20260331_182319.json', 'rb') as file:
#   trino_policies=json.load(file)

# OPTION-2 API Call
url='http://localhost:6080'
auth =  ('admin', 'rangerR0cks')
hive_policies=get_policies_from_ranger(url, auth, 'dev_hive')
trino_policies=get_policies_from_ranger(url, auth, 'dev_trino')

In [ ]:
hive_policies = hive_policies['policies']
trino_policies = trino_policies['policies']

In [ ]:
print(f'Hive Policies:{len(hive_policies)}\nTrino Policies:{len(trino_policies)}')

In [ ]:
hive_policy_names_trino_format = {f'hivesync_{x["id"]}_{x["version"]}':x for x in hive_policies}
trino_policy_names_synced_from_hive = [x['name'] for x in trino_policies if 'hivesync_' in x['name']]

In [ ]:
policies_will_be_inserted = [x for x in hive_policy_names_trino_format.keys() if x not in trino_policy_names_synced_from_hive]

In [ ]:
exclude_list = [6,7,9,10,11,12,68]

In [ ]:
# Remove unsupported and unneccesary policies
policies_will_be_inserted = [x for x in policies_will_be_inserted if int(x.split("_")[1]) not in exclude_list]

In [ ]:
# Detect updated policies using version number
list_hive = hive_policy_names_trino_format.keys() 

hive_formatted = { "_".join(item.split("_")[:-1]): item for item in list_hive }

policies_will_be_updated = []
policies_will_be_deleted = []

for item in trino_policy_names_synced_from_hive:

    item_key = "_".join(item.split("_")[:-1])
    
    if item_key in hive_formatted.keys() and hive_formatted[item_key] != item:
      policies_will_be_updated.append([hive_formatted[item_key], item])
    
    if item_key not in hive_formatted.keys():
      policies_will_be_deleted.append(item)

In [ ]:
policies_will_be_updated

In [ ]:
# Remove updated policies from will be inserted policies
policies_will_be_inserted = [x for x in policies_will_be_inserted if x not in [a[0] for a in policies_will_be_updated] ]

In [ ]:
policies_will_be_inserted

In [ ]:
# Insert Policies
for x in policies_will_be_inserted:
  try:
    hive_policy = RangerPolicy(hive_policy_names_trino_format[x])
    trino_policy =  create_trino_policy_from_hive(hive_policy, 'dev_trino', 'hive')
    ranger.create_policy(trino_policy)
  except Exception as e:
    print(e)
    continue

In [ ]:
# Update Policies
for x in policies_will_be_updated:
  try:
    hive_policy = RangerPolicy(hive_policy_names_trino_format[x[0]])
    trino_policy =  create_trino_policy_from_hive(hive_policy, 'dev_trino', 'hive')
    existing_trino_policy = ranger.get_policy('dev_trino',x[1])
    trino_policy.id = existing_trino_policy.id
    trino_policy.name = x[0]
    ranger.update_policy_by_id(existing_trino_policy.id, trino_policy)
  except NoPolicyItemsError:
    # We don't need the policy anymore.
    policies_will_be_deleted.append(x[1])
    continue
  except UnsupportedPolicyError:
    # We don't need the policy anymore.
    policies_will_be_deleted.append(x[1])
    continue
  except Exception as e:
    print(e)
    continue

In [ ]:
# Delete Policies
for x in policies_will_be_deleted:
  try:
    ranger.delete_policy('dev_trino', x)
  except Exception as e:
    print(e)
    continue